# Межзаписная изменчивость опорного канала и порядок съёмки — диагностика эксперимента EXP_2026_04_29

Цель ноутбука — проверить, какие выводы о порядке записей, дубликатах и
качестве каналов можно получить до физической интерпретации кривой
зондирования. Паспорт серии приведён в
[10.00](10.00_Паспорт_эксперимента_2.md).

Подтверждённый результат: у обоих добровольцев размер сборки уменьшался
от 140 до 50 мм в одном порядке. Поэтому размер, номер записи и время
внутри сессии не варьировались независимо. Канал 1 при неизменном
монтаже заметно менялся между записями, однако причина этой изменчивости
по имеющимся данным не определяется.

Ноутбук не доказывает дрейф прибора, нарушение контакта или неисправность
канала. Эти объяснения рассматриваются ниже только как проверяемые
гипотезы. После реорганизации вычислительные ячейки ожидают принятые
дыхательные разметки серии `11.01`; повторный запуск на внешних данных
ещё не выполнен.

## § 0. Постановка QC и определения

### Наблюдаемые величины

Канал 1 регистрировался неизменным в пределах сессии монтажом ТТРКГ.
Канал 2 регистрировал сменяемую боковую сборку. Неизменность электродов
канала 1 исключает прямое изменение его собственной геометрии, но не
исключает физиологическую вариабельность, изменение контакта,
нестабильность тракта или влияние одновременно работающего канала 2.
Поэтому канал 1 служит индикатором совокупного состояния записи, а не
чистым измерителем времени.

### Термины

- **Межзаписная изменчивость канала 1** — различие `BASE_1` между
  последовательными файлами при неизменной электродной схеме канала 1.
  Термин не задаёт причину различий.
- **Порядок съёмки** — подтверждённая последовательность записей внутри
  сессии. Время последнего изменения файла (`mtime`) используется только
  для проверки согласованности этой последовательности, а не как
  первичный таймстамп эксперимента.
- **QC-флаг канала 1** — отметка необычного уровня `BASE_1`, требующая
  проверки. Флаг не равен диагнозу причины и не исключает автоматически
  одновременно записанный канал 2.
- **QS** — записанный прибором импеданс токовой цепи. В версии МГТУ
  столбцы `QS_1_Ω` и `QS_2_Ω` совпадают; принадлежность рабочего значения
  конкретному каналу не установлена.
- **Отклонение от гладкого хода** — относительный остаток кажущегося
  удельного сопротивления после исследовательского сглаживания по
  размеру. Он не является независимо измеренной ошибкой сборки.

### Метрики изменчивости

Код сообщает три разные описательные величины: изменение от первой к
последней записи в процентах от первой; полный размах в процентах от
медианы; робастный разброс на основе медианного абсолютного отклонения
в процентах от медианы. Эти величины нельзя обозначать одним словом
«дрейф» или взаимозаменять.

Размер и порядок считаются полностью смешанными по дизайну, если каждый
размер измерен один раз и во всех сессиях использован один порядок без
рандомизации или повторов. Это утверждение не требует равенства
коэффициента корреляции точно единице: численное значение корреляции
зависит от неравных временных интервалов и пропущенных размеров.

**Входные данные.** Локальная конфигурация по схеме
`config/exp02_paths.example.json`; внешние CSV эксперимента 2; принятые
дыхательные sidecar-файлы из `11.01` с полным SHA-256 исходной записи,
версией алгоритма и ручным статусом `accepted`.

**Подтверждённые условия протокола.** Сборки записывались последовательно
от 140 до 50 мм. Для Ника файл 100 мм является поздней копией записи
90 мм и не входит в независимый набор. Для Георга 100 мм является
отдельной записью.

**Допущения обработки.** Медиана `BASE` вычисляется внутри принятого
плато задержки дыхания с отступом 0,5 с от границ. Сопоставимость разных
размеров как одного физиологического состояния остаётся допущением.
`mtime` проверяет только согласованность порядка файлов; перенос или
копирование файла может изменить эту метку.

In [ ]:
# @title Внешняя конфигурация и загрузка принятых уровней
import datetime
import hashlib
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

CONFIG_ENV = "KALMYKOV_EXP02_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp02_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
ANNOTATION_DIR = DERIVED_ROOT / "exp02" / "annotations" / "breathing"

subject_items = CONFIG.get("subjects", [])
if len(subject_items) != 2:
    raise ValueError("Для эксперимента 2 конфигурация должна описывать двух добровольцев")

SUBJECTS = {
    item["subject_id"]: {
        "directory": DATA_ROOT / item["data_subdir"],
        "suffix": item["filename_suffix"],
        "label": item.get("report_label", item["subject_id"]),
        "sizes_mm": list(item.get("sizes_mm", [])),
    }
    for item in subject_items
}
for subject_id, info in SUBJECTS.items():
    if not info["sizes_mm"]:
        raise ValueError(f"Для {subject_id} не задан sizes_mm")

HOLD_MARGIN_S = 0.5
QS_CEIL_OHM = 4700.0
REQUIRED_COLUMNS = {
    "TIME_s", "BASE_1_Ω", "BASE_2_Ω", "QS_1_Ω", "QS_2_Ω"
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def accepted_modes(source_path, subject_id, size_mm):
    source_sha256 = sha256_file(source_path)
    annotation_path = ANNOTATION_DIR / f"{source_sha256[:16]}.json"
    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Нет дыхательной разметки 11.01 для {subject_id}, {size_mm} мм"
        )
    annotation = json.loads(annotation_path.read_text(encoding="utf-8"))
    if annotation.get("annotation_type") != "breathing":
        raise ValueError(f"Неверный тип разметки: {annotation_path.name}")
    if annotation.get("input", {}).get("sha256") != source_sha256:
        raise ValueError(f"SHA-256 разметки не совпадает с CSV: {source_path.name}")
    if annotation.get("subject_id") != subject_id or annotation.get("size_mm") != size_mm:
        raise ValueError(f"Разметка относится к другой записи: {annotation_path.name}")
    if annotation.get("qc", {}).get("status") != "accepted":
        raise ValueError(f"Разметка не принята вручную: {annotation_path.name}")
    modes = annotation.get("accepted_modes") or annotation.get("candidate_modes")
    required_modes = {"задержка_вдох", "задержка_выдох"}
    if not modes or not required_modes.issubset(modes):
        raise ValueError(f"В разметке нет двух задержек: {annotation_path.name}")
    return source_sha256, annotation, modes

def level(frame, time_s, span, column):
    start_s, stop_s = map(float, span)
    if stop_s - start_s <= 2 * HOLD_MARGIN_S:
        raise ValueError(f"Слишком короткое плато для {column}: {span}")
    mask = (
        (time_s >= start_s + HOLD_MARGIN_S)
        & (time_s <= stop_s - HOLD_MARGIN_S)
    )
    values = frame.loc[mask, column].to_numpy(dtype=float)
    if values.size == 0 or np.any(~np.isfinite(values)):
        raise ValueError(f"Нет конечных отсчётов {column} внутри плато {span}")
    return float(np.median(values))

RECORDS = {}
for subject_id, info in SUBJECTS.items():
    rows = []
    for size_mm in info["sizes_mm"]:
        source_path = info["directory"] / f"{size_mm}{info['suffix']}.csv"
        if not source_path.exists():
            raise FileNotFoundError(source_path)
        frame = pd.read_csv(source_path, encoding="utf-8")
        missing = REQUIRED_COLUMNS - set(frame.columns)
        if missing:
            raise ValueError(f"В {source_path.name} отсутствуют поля: {sorted(missing)}")
        time_s = frame["TIME_s"].to_numpy(dtype=float)
        delta_t = np.diff(time_s)
        if time_s.size < 2 or np.any(~np.isfinite(time_s)) or np.any(delta_t <= 0):
            raise ValueError(f"TIME_s некорректен в {source_path.name}")

        source_sha256, annotation, modes = accepted_modes(
            source_path, subject_id, size_mm
        )
        stat = source_path.stat()
        rows.append({
            "L": size_mm,
            "mtime": stat.st_mtime,
            "mtime_text": datetime.datetime.fromtimestamp(stat.st_mtime),
            "sha256": source_sha256,
            "b1_in": level(frame, time_s, modes["задержка_вдох"], "BASE_1_Ω"),
            "b1_ex": level(frame, time_s, modes["задержка_выдох"], "BASE_1_Ω"),
            "b2_in": level(frame, time_s, modes["задержка_вдох"], "BASE_2_Ω"),
            "b2_ex": level(frame, time_s, modes["задержка_выдох"], "BASE_2_Ω"),
            "qs_med": level(frame, time_s, modes["задержка_вдох"], "QS_1_Ω"),
            "qs_ceil_pct": 100.0 * float(
                np.mean(frame["QS_1_Ω"].to_numpy(dtype=float) >= QS_CEIL_OHM - 1)
            ),
            "qs_zero_pct": 100.0 * float(
                np.mean(frame["QS_1_Ω"].to_numpy(dtype=float) == 0)
            ),
            "qs_columns_equal": bool(
                np.array_equal(frame["QS_1_Ω"].values, frame["QS_2_Ω"].values)
            ),
            "annotation_version": annotation.get("algorithm_version"),
        })

    rows.sort(key=lambda row: row["mtime"])
    first_mtime = rows[0]["mtime"]
    for order, row in enumerate(rows, 1):
        row["order"] = order
        row["min_from_first_mtime"] = (row["mtime"] - first_mtime) / 60.0
    RECORDS[subject_id] = rows

print(
    "Среда:",
    {"python": sys.version.split()[0], "numpy": np.__version__,
     "pandas": pd.__version__, "matplotlib": plt.matplotlib.__version__},
)
print("Загружено записей:", {name: len(rows) for name, rows in RECORDS.items()})
print(
    "Столбцы QS_1 и QS_2 совпадают во всех записях:",
    "да" if all(row["qs_columns_equal"] for rows in RECORDS.values() for row in rows) else "нет",
)

**Ожидаемый контрольный результат повторного запуска.** Конфигурация
должна загрузить девять независимых записей Ника и десять записей
Георга, проверить полный SHA-256 каждой записи и принять только
разметки `11.01` со статусом `accepted`. Совпадение двух столбцов QS во
всех 19 записях является ранее установленным файловым фактом.

**Статус.** Вычислительные outputs очищены при реорганизации. До нового
запуска на внешних данных приведённые далее численные значения являются
результатами предыдущего проверенного прохода, а не текущего исполнения
этого файла.

## § 1. Порядок съёмки и происхождение дубликата

Подтверждённый автором порядок от 140 до 50 мм проверяется относительно
`mtime` файлов. Контрольная сумма используется для выявления
байт-в-байт совпадающих записей.

**Входные данные.** Полные SHA-256 и время последнего изменения файлов,
рассчитанные в технической ячейке.

**Ограничение.** `mtime` не является временем регистрации и может
измениться при переносе или копировании. Он поддерживает восстановленный
порядок только при согласии с протоколом, соседними файлами и
подтверждением автора. Более поздний `mtime` одной из двух одинаковых
записей сам по себе не устанавливает, когда был получен исходный сигнал.

In [ ]:
# @title § 1. Порядок файлов и полные контрольные суммы
for subject_id, rows in RECORDS.items():
    label = SUBJECTS[subject_id]["label"]
    print(f"=== {label} ===")
    print("  %2s %6s | %19s | %8s | %12s" %
          ("№", "L, мм", "mtime", "мин", "SHA-256"))
    for row in rows:
        print(
            "  %2d %6d | %19s | %8.1f | %12s"
            % (
                row["order"], row["L"],
                row["mtime_text"].strftime("%Y-%m-%d %H:%M:%S"),
                row["min_from_first_mtime"], row["sha256"][:12],
            )
        )
    observed_order = [row["L"] for row in rows]
    expected_order = sorted(SUBJECTS[subject_id]["sizes_mm"], reverse=True)
    print("  mtime согласован с убывающим протоколом:",
          "да" if observed_order == expected_order else "нет")
    print()

print("Проверка дубликатов по всем CSV с указанным суффиксом:")
for subject_id, info in SUBJECTS.items():
    by_sha256 = {}
    for source_path in sorted(info["directory"].glob(f"*{info['suffix']}.csv")):
        digest = sha256_file(source_path)
        by_sha256.setdefault(digest, []).append(source_path.name)
    for digest, names in by_sha256.items():
        if len(names) > 1:
            print(f"  {info['label']}: совпадают {', '.join(names)}; SHA-256={digest}")

**Результат файловой проверки.** Для обоих добровольцев порядок `mtime`
согласован с подтверждённой последовательностью от 140 до 50 мм.
Файлы 90 и 100 мм Ника совпадают по полному SHA-256. Запись 90 мм
находится в подтверждённой последовательности сессии, а файл 100 мм
появился позднее; поэтому независимой считается запись 90 мм. Исходная
отдельная запись 100 мм в доступном наборе отсутствует.

**Следствие для дизайна.** В каждой сессии каждому размеру соответствует
единственный номер записи, а порядок одинаков у обоих добровольцев.
Поэтому размер нельзя отделить от порядка или времени без дополнительных
модельных предположений. Монотонная связь не означает, что Pearson
$r$ с реальными минутами обязан равняться $\pm1$.

Этот результат должен использоваться в
[33.01](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb): для
Ника исключается файл 100 мм, а не запись 90 мм.

## § 2. Межзаписная изменчивость канала 1

Раздел описывает изменение `BASE_1` между файлами при неизменной
электродной схеме канала 1. Изменение не приписывается заранее времени,
контакту или прибору. Геометрия собственного монтажа канала 1 не
менялась, но сборка канала 2 и состояние всей измерительной системы
менялись между записями.

**Входные данные.** Медианные уровни `BASE_1` и `BASE_2` на принятых
задержках вдоха и выдоха, порядок записи и интервалы между `mtime`.

**Допущения.** Принятые дыхательные интервалы соответствуют сравнимым
состояниям. Канал 1 сохранял монтаж в пределах сессии. Возможное
межканальное влияние не устранено, поэтому `BASE_1` характеризует
совокупное состояние измерения, а не изолированный дрейф.

QS используется только как дополнительная записанная величина. Нули,
потолок шкалы и медиана описываются отдельно; до аппаратной проверки они
не объединяются в показатель качества контакта и не служат основанием
для причинного вывода.

In [ ]:
# @title § 2. Три представления данных и однозначные метрики изменчивости
def variability_metrics(values):
    values = np.asarray(values, dtype=float)
    median = float(np.median(values))
    first_last_pct = 100.0 * (values[-1] - values[0]) / values[0]
    full_range_pct_of_median = 100.0 * (values.max() - values.min()) / median
    robust_mad_pct = (
        100.0 * 1.4826 * np.median(np.abs(values - median)) / median
    )
    return {
        "first_last_pct": first_last_pct,
        "full_range_pct_of_median": full_range_pct_of_median,
        "robust_mad_pct": robust_mad_pct,
    }

fig, axes = plt.subplots(len(RECORDS), 3, figsize=(16.5, 4.2 * len(RECORDS)), squeeze=False)
for row_index, (subject_id, rows) in enumerate(RECORDS.items()):
    label = SUBJECTS[subject_id]["label"]
    sizes = np.array([row["L"] for row in rows], dtype=float)
    base1_in = np.array([row["b1_in"] for row in rows])
    base1_ex = np.array([row["b1_ex"] for row in rows])
    order = np.array([row["order"] for row in rows], dtype=float)
    minutes = np.array([row["min_from_first_mtime"] for row in rows])
    qs_median = np.array([row["qs_med"] for row in rows])
    qs_ceiling = np.array([row["qs_ceil_pct"] for row in rows])

    axis = axes[row_index][0]
    qs_axis = axis.twinx()
    qs_axis.bar(sizes, qs_median, width=6.0, color="steelblue", alpha=0.25,
                label="QS, медиана")
    for size, value, ceiling_pct in zip(sizes, qs_median, qs_ceiling):
        if ceiling_pct > 0:
            qs_axis.annotate(
                f"потолок\n{ceiling_pct:.1f}%", (size, value),
                textcoords="offset points", xytext=(0, 3), ha="center", fontsize=6.5,
            )
    qs_axis.set_ylabel("QS, Ом", color="steelblue")
    axis.plot(sizes, base1_in, "o-", color="crimson", label="канал 1, вдох")
    axis.plot(sizes, base1_ex, "s--", color="salmon", label="канал 1, выдох")
    for size, value, record_order in zip(sizes, base1_in, order):
        axis.annotate(str(int(record_order)), (size, value),
                      textcoords="offset points", xytext=(0, 7), ha="center")
    axis.set_xlabel("размер сборки L, мм")
    axis.set_ylabel("BASE_1, Ом")
    axis.set_title(f"{label}: канал 1 и QS по размеру")
    axis.grid(True, alpha=0.3)

    axis = axes[row_index][1]
    axis.plot(minutes, base1_in, "o-", color="crimson", label="вдох")
    axis.plot(minutes, base1_ex, "s--", color="salmon", label="выдох")
    for minute, value, size in zip(minutes, base1_in, sizes):
        axis.annotate(str(int(size)), (minute, value),
                      textcoords="offset points", xytext=(0, 7), ha="center")
    axis.set_xlabel("время от первого mtime, мин")
    axis.set_ylabel("BASE_1, Ом")
    axis.set_title(f"{label}: тот же канал по порядку файлов")
    axis.legend(fontsize=8)
    axis.grid(True, alpha=0.3)

    axis = axes[row_index][2]
    axis.plot(minutes, sizes, "o-", color="navy")
    axis.set_xlabel("время от первого mtime, мин")
    axis.set_ylabel("размер сборки L, мм")
    axis.set_title(f"{label}: порядок размера и файлов")
    axis.grid(True, alpha=0.3)

    metrics = variability_metrics(base1_in)
    print(
        f"{label}: первая→последняя = {metrics['first_last_pct']:+.1f}% от первой; "
        f"полный размах = {metrics['full_range_pct_of_median']:.1f}% от медианы; "
        f"робастный MAD = {metrics['robust_mad_pct']:.1f}% от медианы"
    )
    print(
        f"  описательные r: BASE_1 с номером записи "
        f"{np.corrcoef(order, base1_in)[0, 1]:+.3f}; "
        f"BASE_1 с размером {np.corrcoef(sizes, base1_in)[0, 1]:+.3f}"
    )
    print(
        f"  QS: {qs_median.min():.0f}–{qs_median.max():.0f} Ом; "
        f"r с размером {np.corrcoef(sizes, qs_median)[0, 1]:+.3f}"
    )

plt.tight_layout()
plt.show()

**Разбор прежних процентов.** Для Ника в сохранённом описании указаны
10,79 Ом в первой записи, максимум 13,51 Ом и 12,99 Ом в последней.
Значение около 17 % относилось к прежней паре крайних уровней
10,79→12,66 Ом и устарело после изменения используемого набора.
Изменение от первой к последней записи в текущем описании,
нормированное на первую, составляет около 20,4 %. Значение 25 %
описывает другой объект — рост
от первой записи до максимума. Новый код выводит эти характеристики
раздельно и добавляет полный размах относительно медианы и робастный
разброс.

**Наблюдение.** Канал 1 у Ника меняется между записями плавнее, а у
Георга содержит два резких снижения. Это установленная особенность
записанных значений. Она не определяет причину и не доказывает дрейф
прибора или изменение контакта.

Совпадение столбцов QS является файловым фактом. Связь QS с конкретным
каналом и его пригодность как показателя контакта остаются открытыми.
Малый коэффициент линейной корреляции при 9–10 точках не доказывает
отсутствия зависимости.

## § 3. QC-флаги необычного уровня канала 1

Раздел выделяет записи, которые резко отличаются от остальных значений
того же добровольца. Флаг предназначен для последующей проверки и не
задаёт причину отклонения.

**Входные данные.** `BASE_1` на принятой задержке вдоха во всех записях
каждого добровольца.

**Рабочее правило.** Запись получает QC-флаг, если её уровень меньше
половины или больше удвоенной медианы по добровольцу. Порог является
грубой эвристикой для обнаружения крупных отклонений, а не
метрологически обоснованной границей.

**Гипотезы о причине.** Возможны изменение контакта, кабеля или
положения, особенность регистрации, межканальное влияние, кратковременное
состояние прибора или иная неучтённая причина. Текущие данные не
позволяют выбрать одно объяснение.

In [ ]:
# @title § 3. QC-флаги канала 1
print("%-18s | %5s | %10s | %10s | %s" %
      ("доброволец", "L, мм", "BASE_1", "к медиане", "статус"))
print("-" * 80)
CHANNEL1_FLAGS = {}
for subject_id, rows in RECORDS.items():
    label = SUBJECTS[subject_id]["label"]
    values = np.array([row["b1_in"] for row in rows])
    median = float(np.median(values))
    if median == 0:
        raise ValueError(f"Нулевая медиана BASE_1 у {label}")
    CHANNEL1_FLAGS[subject_id] = []
    for row in rows:
        ratio = row["b1_in"] / median
        flagged = ratio < 0.5 or ratio > 2.0
        if flagged:
            CHANNEL1_FLAGS[subject_id].append(row["L"])
        print(
            "%-18s | %5d | %10.2f | %9.2f× | %s"
            % (label, row["L"], row["b1_in"], ratio,
               "QC-ФЛАГ" if flagged else "")
        )
    print(f"{label}: медиана {median:.2f} Ом; флаги {CHANNEL1_FLAGS[subject_id] or 'нет'}\n")

fig, axes = plt.subplots(1, len(RECORDS), figsize=(6.8 * len(RECORDS), 4.4), squeeze=False)
for index, (subject_id, rows) in enumerate(RECORDS.items()):
    label = SUBJECTS[subject_id]["label"]
    sizes = [row["L"] for row in rows]
    values = [row["b1_in"] for row in rows]
    median = float(np.median(values))
    axis = axes[0][index]
    axis.plot(sizes, values, "o-", color="crimson", label="все записи")
    for size, value in zip(sizes, values):
        if size in CHANNEL1_FLAGS[subject_id]:
            axis.plot(size, value, "X", ms=14, color="black", label="QC-флаг")
    axis.axhline(median, color="0.5", ls="--", lw=1, label=f"медиана {median:.1f} Ом")
    axis.axhspan(0.5 * median, 2.0 * median, color="green", alpha=0.08)
    axis.set_xlabel("размер сборки L, мм")
    axis.set_ylabel("BASE_1, Ом")
    axis.set_title(f"{label}: QC канала 1")
    handles, labels = axis.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    axis.legend(unique.values(), unique.keys(), fontsize=8)
    axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Наблюдение предыдущего запуска.** Для Ника крупных отклонений по
рабочему порогу не найдено. У Георга записи 90 и 100 мм имеют уровни
канала 1 около 0,20 и 0,31 медианы и получают QC-флаги.

**Интерпретация.** Установлен только необычно низкий уровень канала 1.
Нарушение контакта является одной из гипотез, но не подтверждённой
причиной. Записи нельзя автоматически исключать по каналу 2: его
пригодность проверяется отдельно.

QS даёт несогласованную картину: запись 100 мм также необычна по QS, а
запись 90 мм не выделяется тем же способом. Поэтому QS и `BASE_1` нельзя
объединять в единый критерий до аппаратной расшифровки.

## § 4. Чувствительность остаточного анализа к QC-флагам

Раздел проверяет, насколько описательная согласованность отклонений
кривой зондирования меняется после исключения только отмеченных записей
соответствующего добровольца.

**Входные данные.** Уровни `BASE_1` и `BASE_2` на задержке вдоха и
QC-флаги § 3.

**Исследовательское определение.** Для каждого добровольца вычисляется
кажущееся удельное сопротивление, затем по тем же точкам подбирается
многочлен второй степени по логарифму размера. Относительный остаток
обозначается $\delta$. Выбор сглаживания эвристический, поэтому
корреляции остатков являются описательными и не образуют независимый
статистический тест физической причины.

**Проверяемый вопрос.** Сохраняется ли видимая согласованность остатков,
если из кривой каждого добровольца удалить только его собственные
QC-флаги? Исключение выполняется как анализ чувствительности, а не как
окончательная выбраковка канала 2.

In [ ]:
# @title § 4. Остатки и раздельное исключение QC-флагов
SMOOTHING_DEGREE = 2

def deviations(rows, drop_sizes=()):
    selected = [row for row in rows if row["L"] not in set(drop_sizes)]
    if len(selected) <= SMOOTHING_DEGREE:
        raise ValueError("Недостаточно точек для сглаживания")
    sizes = np.array([row["L"] for row in selected], dtype=float)
    a = sizes / 2 / 1000.0
    b = a / 2
    apparent_rho = (
        np.array([row["b2_in"] for row in selected])
        * np.pi * (a ** 2 - b ** 2) / (2 * b)
    )
    coefficients = np.polyfit(np.log(sizes), apparent_rho, SMOOTHING_DEGREE)
    smooth = np.polyval(coefficients, np.log(sizes))
    delta_pct = 100.0 * (apparent_rho - smooth) / smooth
    base1 = np.array([row["b1_in"] for row in selected])
    return sizes, delta_pct, base1

subject_ids = list(RECORDS)
if len(subject_ids) != 2:
    raise ValueError("Межсубъектное сравнение рассчитано для двух добровольцев")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for subject_id in subject_ids:
    sizes, delta_pct, base1 = deviations(RECORDS[subject_id])
    correlation = np.corrcoef(base1, delta_pct)[0, 1]
    label = SUBJECTS[subject_id]["label"]
    print(
        f"{label}: описательная корреляция δ с BASE_1 "
        f"r={correlation:+.3f}, n={len(delta_pct)}"
    )
    axes[0].plot(base1, delta_pct, "o", label=f"{label}, r={correlation:+.2f}")
axes[0].axhline(0, color="0.6", lw=0.8)
axes[0].set_xlabel("BASE_1, Ом")
axes[0].set_ylabel("остаток δ, %")
axes[0].set_title("Описательная связь остатка с каналом 1")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

scenarios = [
    ("все записи", {subject_id: () for subject_id in subject_ids}, "o-"),
    ("раздельно без QC-флагов", CHANNEL1_FLAGS, "s--"),
]
for scenario_name, drop_by_subject, style in scenarios:
    residuals = {}
    for subject_id in subject_ids:
        sizes, delta_pct, _ = deviations(
            RECORDS[subject_id], drop_sizes=drop_by_subject.get(subject_id, ())
        )
        residuals[subject_id] = dict(zip(sizes, delta_pct))

    common_sizes = sorted(
        set(residuals[subject_ids[0]]) & set(residuals[subject_ids[1]])
    )
    first = np.array([residuals[subject_ids[0]][size] for size in common_sizes])
    second = np.array([residuals[subject_ids[1]][size] for size in common_sizes])
    correlation = np.corrcoef(first, second)[0, 1]
    sign_agreement = int(np.sum(np.sign(first) == np.sign(second)))
    print(
        f"{scenario_name}: общих размеров {len(common_sizes)}, "
        f"r={correlation:+.3f}, совпадение знака "
        f"{sign_agreement}/{len(common_sizes)}"
    )
    for subject_id, values in zip(subject_ids, (first, second)):
        label = SUBJECTS[subject_id]["label"]
        axes[1].plot(
            common_sizes, values, style,
            label=f"{label}: {scenario_name}",
        )

axes[1].axhline(0, color="0.6", lw=0.8)
axes[1].set_xlabel("размер сборки L, мм")
axes[1].set_ylabel("остаток δ, %")
axes[1].set_title("Чувствительность к раздельным QC-флагам")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Статус прежних чисел.** Ранее после «исключения дефектных записей»
один и тот же список размеров удалялся из кривых обоих добровольцев.
Поэтому опубликованное в старом тексте значение корреляции после
исключения не является результатом требуемой проверки и не сохраняется
как вывод.

Исправленный код удаляет у каждого добровольца только его собственные
QC-флаги, затем сравнивает общие размеры. Результат должен быть получен
повторным запуском. Даже после запуска коэффициент корреляции и
совпадение знаков останутся описательными: остатки построены на тех же
точках, объём выборки мал, а размер полностью смешан с порядком.

Допустимый текущий вывод: простая линейная связь остатка $\delta$ с
уровнем канала 1 ранее не обнаружена; причина отклонений кривой
зондирования не установлена.

## § 4.1. Гипотеза межканального влияния

**Наблюдение из эксперимента 3.** На реокардиомониторе версии РНЦХ
показания оставшегося подключённого канала изменялись при отключении
кабеля другого канала без снятия электродов. Установлен сам факт
зависимости показаний от состояния второго канала. Физическая причина
эффекта в том эксперименте не определялась.

**Гипотеза для эксперимента 2.** Аналогичная зависимость может
существовать в версии МГТУ и меняться с геометрией боковой сборки.
Возможные объяснения включают взаимодействие измерительных цепей,
изменение нагрузки, особенности коммутации или алгоритма прибора.
Представление о параллельном проводящем пути является одной из моделей,
а не установленным механизмом. Числа версии РНЦХ не переносятся на
версию МГТУ как поправка.

QS не исключает контакт и не подтверждает межканальное влияние: в
версии МГТУ два столбца совпадают, а принадлежность рабочего значения
неизвестна.

**Проверка гипотезы.** В протоколе `10.02` следует для каждого размера
выполнить повторные парные измерения канала 2 при работающем и
отключённом канале 1, не меняя электроды внутри пары. Порядок состояний
и размеров следует рандомизировать и повторить. Разность характеризует
суммарный эффект переключения в заданных условиях, но сама по себе не
устанавливает его физический механизм.

## § 4.2. Гипотеза о неправильной абсолютной шкале канала 1

**Наблюдаемая проблема.** Для сходной схемы ТТРКГ в документах указаны
разные порядки базового импеданса: около 12 Ом для версии МГТУ,
54 Ом для версии РНЦХ при двух работающих каналах и 93–101 Ом для
версии РНЦХ при отключённом канале 2. Исторический COMSOL-расчёт даёт
значение порядка 100 Ом.

В МГТУ и РНЦХ участвовал один доброволец Ник, но записи выполнены в
разные дни, разными версиями прибора и с погрешностью повторной ручной
установки электродов. Общая схема монтажа одинакова, однако точное
совпадение координат и контактных условий не подтверждено.

**Ограничение расчётного сравнения.** Значение `100+ Ом` относится к
исторической COMSOL/FEM-модели. Полные входы, точная геометрия,
проводимости, граничные условия и воспроизводимый запуск в текущем
Git-состоянии не восстановлены. Модель не является метрологическим
эталоном и не доказывает исправность одной версии прибора.

**Гипотеза.** Абсолютный масштаб канала 1 версии МГТУ может быть
неправильным или несопоставимым с версией РНЦХ. Конкурирующие объяснения:
различия gain и фильтрации, межканальное влияние, контакт и геометрия,
различия сессий, а также ошибка исторической модели. До прямой
калибровки нельзя объявлять канал неисправным или считать расхождение
объяснённым только эталоном сравнения.

**Следствие для текущего QC.** `BASE_1` используется как записанный
индикатор межзаписной изменчивости. Сохранение физического относительного
масштаба также является допущением и должно быть проверено. План прямой
калибровки и сравнения версий относится к `10.02`.

## § 5. Выводы и границы результата

### Подтверждённые факты

1. У обоих добровольцев размеры регистрировались один раз в одинаковом
   порядке от 140 до 50 мм. Размер, номер записи и время сессии не
   варьировались независимо.
2. У Ника файлы 90 и 100 мм совпадают; независимой является запись
   90 мм, а отдельная запись 100 мм в доступном наборе отсутствует.
3. Канал 1 меняется между файлами при неизменной собственной схеме
   электродов. Это межзаписная изменчивость с неустановленной причиной.
4. Записи Георга 90 и 100 мм получают QC-флаги по необычно низкому
   уровню канала 1. Это не является автоматическим основанием для
   исключения канала 2.
5. Столбцы `QS_1_Ω` и `QS_2_Ω` совпадают во всех 19 доступных файлах.

### Допущения и нерешённые вопросы

- Сравнение размеров предполагает воспроизводимость физиологического
  состояния между последовательными записями.
- Причина QC-флагов не установлена; контакт является одной из гипотез.
- Межканальное влияние в версии МГТУ и неправильный абсолютный масштаб
  канала 1 являются отдельными гипотезами, требующими протокола `10.02`.
- Причина остаточных отклонений кривой зондирования не установлена.

### Требования к продолжению

1. Исправить и принять вручную все 19 дыхательных sidecar-файлов
   `11.01`, затем повторно выполнить настоящий ноутбук.
2. Сохранить производный QC-манифест с SHA-256, версией разметки,
   метриками и флагами, не включая исходные медицинские данные.
3. В новом эксперименте рандомизировать порядок размеров, протоколировать
   фактическое время и выполнять повторные контрольные измерения.
4. Проверку межканального влияния и абсолютного масштаба выполнять по
   отдельному калибровочному протоколу `10.02`.